# Блокнот 03: Географічна типізація та ієрархічна кластеризація (RQ2)

**Дослідницьке питання (RQ2):**
> *Як принципово відрізняються профілі загроз для різних регіонів (глибокий тил на зразок Закарпаття, транзитний центр на зразок Київщини, прифронтові східні та південні області), і на яку оптимальну кількість кластерів $K$ ділиться безпекова карта України на основі дендрограми та метрик якості кластеризації (Silhouette Score, Davies-Bouldin)?*

У цьому блокноті реалізовано:
1. Побудову 5-вимірного цифрового паспорта для 24 регіонів України.
2. Z-стандартизацію ознак через `StandardScaler`.
3. Пошук оптимального $K \in [2, 6]$ за метриками Silhouette Score та Davies-Bouldin Index.
4. Ієрархічну агломеративну кластеризацію за методом Варда (Ward linkage).
5. Візуалізацію дендрограми та аналіз відмінностей між отриманими кластерами.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

# Завантажуємо зведену таблицю профілів кластеризації
summary_path = Path("../reports/regional_clusters_summary.csv")
profiles_df = pd.read_csv(summary_path, index_col="oblast")
print(f"Завантажено профілі для {len(profiles_df)} регіонів України.")
display(profiles_df.head())

## 1. Математичне обґрунтування вибору кількості кластерів (K)

Ми не фіксуємо кількість кластерів наосліп. Перебір $K \in [2, 6]$ оцінюється за двома незалежними метриками якості:
- **Silhouette Score (Коефіцієнт силуету):** показує якість відокремленості кластерів (максимізуємо).
- **Davies-Bouldin Index:** відношення внутрішньокластерного розсіювання до міжкластерної відстані (мінімізуємо).

In [ ]:
display(Image("../reports/figures/04_clustering_metrics_evaluation.png"))

## 2. Ієрархічна дендрограма зв'язків між областями (Ward Linkage)

Дендрограма наочно відображає «генеалогічне дерево» схожості областей України за характером загроз.

In [ ]:
display(Image("../reports/figures/05_regional_dendrogram.png"))

## 3. Зведена аналітична характеристика виділених кластерів

In [ ]:
cluster_summary = profiles_df.groupby("cluster_name")[
    ["alerts_per_month", "median_duration_min", "night_ratio", "short_ratio", "long_ratio"]
].mean()

# Переводимо частки у відсотки для наочності
cluster_summary["night_ratio_%"] = cluster_summary["night_ratio"] * 100
cluster_summary["short_ratio_%"] = cluster_summary["short_ratio"] * 100
cluster_summary["long_ratio_%"] = cluster_summary["long_ratio"] * 100
display(cluster_summary[["alerts_per_month", "median_duration_min", "night_ratio_%", "short_ratio_%", "long_ratio_%"]])

## 4. Склад кожного кластера

In [ ]:
for c_name, group in profiles_df.groupby("cluster_name"):
    print(f"\n📍 {c_name.upper()} ({len(group)} областей):")
    print(", ".join(sorted(group.index.tolist())))
    print(f"Середня кількість тривог на місяць: {group['alerts_per_month'].mean():.1f}")